**Experiment 1 – baseline. See notebook 10 for the honest evaluation.**

In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix,
    classification_report)

from imblearn.over_sampling import SMOTE

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

import joblib

In [16]:
churn_features = pd.read_csv("../data/processed/customer_features.csv")

print("Shape:", churn_features.shape)

churn_features.head()

Shape: (77546, 11)


,user_id,total_orders,total_spend,total_quantity,average_order_value,first_order_date,last_order_date,age,gender,marital_status,occupation
0,1,2,1672,5,836.000000,2017-12-12,2018-11-14,20,Female,Single,Student
1,2,3,4061,7,1353.666667,2018-02-13,2018-08-03,24,Female,Single,Student
2,3,1,21542,500,21542.000000,2019-05-15,2019-05-15,22,Male,Single,Student
3,4,1,102,1,102.000000,2019-02-06,2019-02-06,22,Female,Single,Student
4,5,2,52047,50,26023.500000,2018-03-02,2018-07-25,22,Male,Single,Student


PREPARING THE CHURN DATASET

In [17]:
# Convert date columns
churn_features["first_order_date"] = pd.to_datetime(churn_features["first_order_date"])

churn_features["last_order_date"] = pd.to_datetime(churn_features["last_order_date"])

# Define churn cutoff
cutoff_date = pd.Timestamp("2019-12-31")

print("Cutoff date:", cutoff_date)

Cutoff date: 2019-12-31 00:00:00


LOADING THE ORDERS DATA

In [18]:
orders = pd.read_csv("../data/processed/orders_cleaned.csv")

orders["order_date"] = pd.to_datetime(orders["order_date"])

print("Orders shape:", orders.shape)
print("Date range:")
print("Start:", orders["order_date"].min())
print("End:", orders["order_date"].max())

Orders shape: (148528, 6)
Date range:
Start: 2017-10-04 00:00:00
End: 2020-06-26 00:00:00


SPLIT HISTORICAL AND FURTURE ORDERSm

In [19]:
historical_orders = orders[orders["order_date"] <= cutoff_date].copy()

future_orders = orders[orders["order_date"] > cutoff_date].copy()

print("Historical orders:", historical_orders.shape)
print("Future orders:", future_orders.shape)

Historical orders: (127165, 6)
Future orders: (21363, 6)


CREATING HISTORICAL CUSTOMER FEATURES

In [20]:
churn_features = historical_orders.groupby("user_id").agg(
    total_orders=("user_id", "count"),
    total_spend=("sales_amount", "sum"),
    total_quantity=("sales_qty", "sum"),
    first_order_date=("order_date", "min"),
    last_order_date=("order_date", "max")
).reset_index()

print("Customer feature shape:", churn_features.shape)

churn_features.head()

Customer feature shape: (72069, 6)


,user_id,total_orders,total_spend,total_quantity,first_order_date,last_order_date
0,1,2,1672,5,2017-12-12,2018-11-14
1,2,3,4061,7,2018-02-13,2018-08-03
2,3,1,21542,500,2019-05-15,2019-05-15
3,4,1,102,1,2019-02-06,2019-02-06
4,5,2,52047,50,2018-03-02,2018-07-25


CALCULATING AVERAGE ORDER VALUE

In [21]:
churn_features["average_order_value"] = (churn_features["total_spend"] /
                                         churn_features["total_orders"])

churn_features[["user_id","total_orders","total_spend","average_order_value"]].head()

,user_id,total_orders,total_spend,average_order_value
0,1,2,1672,836.000000
1,2,3,4061,1353.666667
2,3,1,21542,21542.000000
3,4,1,102,102.000000
4,5,2,52047,26023.500000


CALCULATING RECENCY

In [22]:
churn_features["recency"] = (cutoff_date - churn_features["last_order_date"]).dt.days

churn_features[[ "user_id","last_order_date","recency"]].head()

,user_id,last_order_date,recency
0,1,2018-11-14,412
1,2,2018-08-03,515
2,3,2019-05-15,230
3,4,2019-02-06,328
4,5,2018-07-25,524


CREATING CHURN TARGET

In [23]:
future_customers = set(future_orders["user_id"].unique())

churn_features["churn"] = (~churn_features["user_id"].isin(future_customers)).astype(int)

churn_features[["user_id", "churn"]].head()

,user_id,churn
0,1,1
1,2,1
2,3,1
3,4,1
4,5,1


CHECKING CHURN DISTRIBUTION

In [24]:
print(churn_features["churn"].value_counts())

print("\nChurn proportion:")
print(churn_features["churn"].value_counts(normalize=True))

churn
1    58287
0    13782
Name: count, dtype: int64

Churn proportion:
churn
1    0.808767
0    0.191233
Name: proportion, dtype: float64


CREATING ADDITIONAL BEHAVIORAL FEATURES

In [25]:
churn_features["tenure_days"] = (churn_features["last_order_date"]- churn_features["first_order_date"]).dt.days

churn_features["active_months"] = (churn_features["tenure_days"] / 30).round(2)

churn_features["average_days_between_orders"] = np.where(churn_features["total_orders"] > 1,churn_features["tenure_days"] / (
    churn_features["total_orders"] - 1),0)

churn_features["orders_per_month"] = np.where(churn_features["active_months"] > 0,churn_features["total_orders"] /
    churn_features["active_months"],churn_features["total_orders"])

churn_features[["user_id","tenure_days","active_months","average_days_between_orders","orders_per_month"]].head()

,user_id,tenure_days,active_months,average_days_between_orders,orders_per_month
0,1,337,11.23,337.0,0.178094
1,2,171,5.70,85.5,0.526316
2,3,0,0.00,0.0,1.000000
3,4,0,0.00,0.0,1.000000
4,5,145,4.83,145.0,0.414079


SELECT FEATURES AND TARGET

In [26]:
X = churn_features[["total_orders","total_spend","total_quantity","average_order_value","recency",
        "tenure_days","active_months","average_days_between_orders","orders_per_month"]]

y = churn_features["churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

X shape: (72069, 9)
y shape: (72069,)

Features:
['total_orders', 'total_spend', 'total_quantity', 'average_order_value', 'recency', 'tenure_days', 'active_months', 'average_days_between_orders', 'orders_per_month']


TRAIN - TEST - SPLIT

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train distribution:")
print(y_train.value_counts())

print("\ny_test distribution:")
print(y_test.value_counts())

X_train shape: (57655, 9)
X_test shape: (14414, 9)

y_train distribution:
churn
1    46629
0    11026
Name: count, dtype: int64

y_test distribution:
churn
1    11658
0     2756
Name: count, dtype: int64


STANDARD SCALER

In [28]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (57655, 9)
X_test_scaled shape: (14414, 9)


In [34]:
early_stopping = EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True)

In [35]:
X_train_nn, X_val_nn, y_train_nn, y_val_nn = train_test_split(X_train_scaled,y_train,test_size=0.20,random_state=42,
    stratify=y_train)

print("Training:", X_train_nn.shape)
print("Validation:", X_val_nn.shape)

print("\nTraining distribution:")
print(y_train_nn.value_counts())

print("\nValidation distribution:")
print(y_val_nn.value_counts())

Training: (46124, 9)
Validation: (11531, 9)

Training distribution:
churn
1    37303
0     8821
Name: count, dtype: int64

Validation distribution:
churn
1    9326
0    2205
Name: count, dtype: int64


In [36]:
smote = SMOTE(random_state=42)

X_train_nn_smote, y_train_nn_smote = smote.fit_resample(
    X_train_nn,
    y_train_nn
    
)

print("Before SMOTE:")
print(y_train_nn.value_counts())

print("\nAfter SMOTE:")
print(y_train_nn_smote.value_counts())

print("\nTraining shape after SMOTE:")
print(X_train_nn_smote.shape)

Before SMOTE:
churn
1    37303
0     8821
Name: count, dtype: int64

After SMOTE:
churn
1    37303
0    37303
Name: count, dtype: int64

Training shape after SMOTE:
(74606, 9)


REBUILD THE NEURAL NETWORK

In [37]:
model = Sequential([
    Dense(32, activation="relu", input_shape=(X_train_nn_smote.shape[1],)),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

C:\Users\ASUS\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                      │ (None, 32)                  │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 865 (3.38 KB)

 Trainable params: 865 (3.38 KB)

 Non-trainable params: 0 (0.00 B)

TRAIN THE NEURAL NETWORK

In [38]:
early_stopping = EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True)

history = model.fit(X_train_nn_smote,y_train_nn_smote,validation_data=(X_val_nn, y_val_nn),epochs=20,
    batch_size=32,callbacks=[early_stopping],verbose=1)

Epoch 1/20
2332/2332 ━━━━━━━━━━━━━━━━━━━━ 14s 5ms/step - accuracy: 0.5056 - loss: 0.6939 - val_accuracy: 0.5795 - val_loss: 0.6854
Epoch 2/20
2332/2332 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - accuracy: 0.5092 - loss: 0.6929 - val_accuracy: 0.6019 - val_loss: 0.6929
Epoch 3/20
2332/2332 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.5136 - loss: 0.6927 - val_accuracy: 0.5767 - val_loss: 0.6836
Epoch 4/20
2332/2332 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.5157 - loss: 0.6925 - val_accuracy: 0.4227 - val_loss: 0.7044
Epoch 5/20
2332/2332 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.5162 - loss: 0.6922 - val_accuracy: 0.4805 - val_loss: 0.6960
Epoch 6/20
2332/2332 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.5168 - loss: 0.6920 - val_accuracy: 0.4811 - val_loss: 0.6954


In [39]:
y_pred_proba = model.predict(X_test_scaled).ravel()

y_pred = (y_pred_proba >= 0.5).astype(int)

print("Predictions generated successfully.")
print("Prediction shape:", y_pred.shape)

451/451 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Predictions generated successfully.
Prediction shape: (14414,)


CALCULATE NEURAL NETWORK METRICS

In [40]:
accuracy = accuracy_score(y_test, y_pred)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

print("Neural Network Results")
print("----------------------")
print("Accuracy:", accuracy)
print("Balanced Accuracy:", balanced_accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)

Neural Network Results
----------------------
Accuracy: 0.5824198695712501
Balanced Accuracy: 0.5010801306016835
Precision: 0.8093252879868349
Recall: 0.6327843540916109
F1 Score: 0.7102488807586771
ROC-AUC: 0.49965262397287374
PR-AUC: 0.8070108452268071


CONFUSION MATRIX

In [41]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test,y_pred))

Confusion Matrix:
[[1018 1738]
 [4281 7377]]

Classification Report:
              precision    recall  f1-score   support

           0       0.19      0.37      0.25      2756
           1       0.81      0.63      0.71     11658

    accuracy                           0.58     14414
   macro avg       0.50      0.50      0.48     14414
weighted avg       0.69      0.58      0.62     14414



In [42]:
model.save("../models/churn_neural_network.keras")

print("Neural Network model saved successfully.")

Neural Network model saved successfully.
